In [3]:
!pip install langchain langchain_openai mem0ai

  Using cached langchain_openai-0.3.6-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached orjson-3.10.15-cp312-cp312-macosx_10_15_x86_64.macosx_11_0_arm64.macosx_10_15_universal2.whl.metadata (41 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached zstandard-0.23.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (3.0 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 177.1 kB/s eta 0:00:00a 0:00:02
Using cached langchain_openai-0.3.6-py3-none-any.whl (54 kB)
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached orjson-3.10.15-cp312-cp312-macosx_10_15_x86_64.macosx_11_0_arm64.macosx_10_15_universal2.whl (249 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached zstandard-0.23.0-cp312-cp312-macosx_11_0_arm64.whl (633 kB)
Using cached jsonpointer-3.0.0-py2.py3-no

In [4]:
import os
from typing import List, Dict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from mem0 import MemoryClient

# Configuration
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["MEM0_API_KEY"] = "m0-tRT3al4pCb4YWgfkQCUuK1x67bLi7YjJsLj3Tyhz"

# Initialize LangChain and Mem0
llm = ChatOpenAI(model="gpt-4o-mini")
mem0 = MemoryClient(api_key=os.environ["MEM0_API_KEY"])

In [5]:
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="""You are a helpful travel agent AI. Use the provided context to personalize your responses and remember user preferences and past interactions. 
    Provide travel recommendations, itinerary suggestions, and answer questions about destinations. 
    If you don't have specific information, you can make general suggestions based on common travel knowledge."""),
    MessagesPlaceholder(variable_name="context"),
    HumanMessage(content="{input}")
])

In [6]:
def retrieve_context(query: str, user_id: str) -> List[Dict]:
    """Retrieve relevant context from Mem0"""
    memories = mem0.search(query, user_id=user_id)
    seralized_memories = ' '.join([mem["memory"] for mem in memories])
    context = [
        {
            "role": "system", 
            "content": f"Relevant information: {seralized_memories}"
        },
        {
            "role": "user",
            "content": query
        }
    ]
    return context

def generate_response(input: str, context: List[Dict]) -> str:
    """Generate a response using the language model"""
    chain = prompt | llm
    response = chain.invoke({
        "context": context,
        "input": input
    })
    return response.content

def save_interaction(user_id: str, user_input: str, assistant_response: str):
    """Save the interaction to Mem0"""
    interaction = [
        {
          "role": "user",
          "content": user_input
        },
        {
            "role": "assistant",
            "content": assistant_response
        }
    ]
    mem0.add(interaction, user_id=user_id)

In [8]:
def chat_turn(user_input: str, user_id: str) -> str:
    # Retrieve context
    context = retrieve_context(user_input, user_id)
    
    # Generate response
    response = generate_response(user_input, context)
    
    # Save interaction
    save_interaction(user_id, user_input, response)
    
    return response

In [9]:
if __name__ == "__main__":
    print("Welcome to your personal Travel Agent Planner! How can I assist you with your travel plans today?")
    user_id = "john"
    
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("Travel Agent: Thank you for using our travel planning service. Have a great trip!")
            break
        
        response = chat_turn(user_input, user_id)
        print(f"Travel Agent: {response}")

Welcome to your personal Travel Agent Planner! How can I assist you with your travel plans today?


/Users/ngocp/Documents/projects/homework-ai-helper/.venv/lib/python3.12/site-packages/mem0/client/main.py:32: DeprecationWarning: Using default output format 'v1.0' is deprecated and will be removed in version 0.1.70. Please use output_format='v1.1' for enhanced memory details. Check out the docs for more information: https://docs.mem0.ai/platform/quickstart#4-1-create-memories
  return func(*args, **kwargs)


Travel Agent: Traveling to Da Nang during the Tet holiday sounds exciting! Tet, the Vietnamese Lunar New Year, is a vibrant time to visit, with festive celebrations, beautiful decorations, and delicious traditional food. Here are some personalized recommendations for your trip:

### Recommended Itinerary for Tet in Da Nang:

**Day 1: Arrival and Local Exploration**
- Arrive in Da Nang and check into your hotel.
- Head to the Han River to see the beautiful flower streets and enjoy the festive atmosphere.
- Visit the Dragon Bridge, which often has dragon fire shows in the evenings—check the schedule!

**Day 2: Cultural Highlights**
- Visit the Marble Mountains for breathtaking views and explore the caves and pagodas.
- Enjoy a traditional Tet market where you can find unique local gifts and snacks.
- In the evening, enjoy a Tet feast at a local restaurant—try traditional dishes like Bánh Chưng and sticky rice.

**Day 3: Beach Day**
- Spend the day relaxing at My Khe Beach, known for its 

/Users/ngocp/Documents/projects/homework-ai-helper/.venv/lib/python3.12/site-packages/mem0/client/main.py:32: DeprecationWarning: Using default output format 'v1.0' is deprecated and will be removed in version 0.1.70. Please use output_format='v1.1' for enhanced memory details. Check out the docs for more information: https://docs.mem0.ai/platform/quickstart#4-1-create-memories
  return func(*args, **kwargs)


Travel Agent: Traveling from Hanoi to Da Nang during the Tet holiday can vary in price depending on your mode of transport and how early you book. Here are some general estimates:

1. **Flight**: Prices for a round-trip flight from Hanoi to Da Nang typically range from $50 to $150, depending on how close you are to the departure date and if you book in advance. During Tet, prices may be on the higher end due to increased demand.

2. **Train**: Taking a train might cost between $20 and $60 for a one-way ticket, depending on the class of service you choose. Trains often get crowded during Tet, so it's advisable to book early.

3. **Bus**: A bus ticket can range from $15 to $30 for one way. This is usually the most budget-friendly option, but it may take longer than flying or taking the train.

For the best experience, I recommend booking your transportation as soon as possible due to the heavy travel traffic during the Tet holiday. Would you like assistance with booking or finding accomm

...

Travel Agent: Chào bạn! Bạn muốn đi từ đâu đến Đà Nẵng bằng máy bay và xe buýt? Sau khi tới Đà Nẵng, bạn có kế hoạch khám phá những địa điểm nào không? Tôi có thể giúp bạn lên kế hoạch cho chuyến đi trong 3 ngày 2 đêm vào dịp Tết này!


/Users/ngocp/Documents/projects/homework-ai-helper/.venv/lib/python3.12/site-packages/mem0/client/main.py:32: DeprecationWarning: Using default output format 'v1.0' is deprecated and will be removed in version 0.1.70. Please use output_format='v1.1' for enhanced memory details. Check out the docs for more information: https://docs.mem0.ai/platform/quickstart#4-1-create-memories
  return func(*args, **kwargs)


Travel Agent: That sounds like a fantastic trip! Here’s a suggested itinerary for your 3 days and 2 nights in Da Nang, including visits to Bà Nà Hills and Hội An, especially during Tet:

### Day 1: Arrival in Da Nang
- **Morning**: 
  - Fly from Hà Nội to Da Nang. 
  - Check into your hotel and freshen up.
- **Afternoon**: 
  - Head to Bà Nà Hills. The cable car ride offers stunning views of the mountains and sea. Spend time exploring the Golden Bridge and the Linh Ung Pagoda.
- **Evening**: 
  - Return to Da Nang and enjoy a welcome dinner at a local restaurant. Try some traditional Vietnamese cuisine like Mi Quang or seafood.

### Day 2: Explore Hội An
- **Morning**: 
  - After breakfast, take a short ride (about 30 minutes) to Hội An.
  - Explore the ancient town, stroll through the lantern-lit streets, and visit historical sites like the Japanese Covered Bridge.
- **Lunch**: 
  - Enjoy lunch at a riverside restaurant. Try some local specialties such as Cao Lau.
- **Afternoon**: 
  

HTTP error occurred: Client error '400 Bad Request' for url 'https://api.mem0.ai/v1/memories/search/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400


APIError: API request failed: {"query":["This field may not be blank."]}